# Audit Standards Anthropic RAG Pipeline Test

Phase 4에서 적재한 4개 Qdrant collection을 대상으로 검색 결과를 Anthropic Messages API 컨텍스트로 조합하고, 답변과 citation을 확인하는 테스트 notebook입니다.

| Step | 설명 | 데이터 소스 |
|---:|---|---|
| 1 | 기준서 식별 | Qdrant `summary` named vector |
| 2 | 문단 검색 | Qdrant `passage` named vector + `standard_id` filter |
| 3 | 컨텍스트 구성 | Qdrant payload `content_text`, `heading_trail`, `chunk_id` |
| 4 | LLM 답변 | Anthropic Messages API |
| 5 | 결과 저장 | `output/phase4_llm_rag_results.json` |

사전 조건:

- Qdrant가 `http://localhost:6333`에서 실행 중이어야 합니다.
- `.env`에 `UPSTAGE_API_KEY`가 있어야 query embedding을 만들 수 있습니다.
- LLM 답변까지 실행하려면 `.env`에 `ANTHROPIC_API_KEY`를 설정하세요.
- 모델은 `ANTHROPIC_MODEL`로 바꿀 수 있습니다. 기본값은 `claude-sonnet-4-20250514`입니다.

중요: 이전 버전 notebook을 실행한 커널에는 OpenAI/auto-provider 함수가 남아 있을 수 있습니다. 이 파일로 교체한 뒤에는 `Kernel > Restart Kernel and Run All Cells`로 다시 실행하세요.


## 1. 환경 설정

프로젝트 루트, `.env`, Qdrant client, Upstage query embedder를 초기화합니다.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import httpx
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

from audit_parser.ingest.embedder import Embedder
from audit_parser.ingest.qdrant_writer import (
    KIND_STANDARD_SUMMARY,
    VECTOR_PASSAGE,
    VECTOR_SUMMARY,
)

load_dotenv(PROJECT_ROOT / ".env")

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY") or None
CACHE_PATH = PROJECT_ROOT / ".embed_cache.sqlite"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=30)
embedder = Embedder(cache_path=CACHE_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Qdrant:      {QDRANT_URL}")
print(f"Cache:       {CACHE_PATH} (exists={CACHE_PATH.exists()})")


Project root: /home/shin/Project/_AuditStandard_parsing
Qdrant:      http://localhost:6333
Cache:       /home/shin/Project/_AuditStandard_parsing/.embed_cache.sqlite (exists=True)


## 2. Config

아래 상수만 수정해도 검색 범위와 Anthropic 호출 동작을 바꿀 수 있습니다.

In [2]:
COLLECTIONS: dict[str, dict[str, Any]] = {
    "audit_standards_회계감사기준_2025": {
        "label": "ISA 36종",
        "expected_points": 8626,
        "expected_summary_points": 36,
    },
    "audit_standards_품질관리기준서_2018": {
        "label": "ISQM-1",
        "expected_points": 400,
        "expected_summary_points": 1,
    },
    "audit_standards_기타인증업무기준_2022": {
        "label": "ASSR-3000",
        "expected_points": 1240,
        "expected_summary_points": 1,
    },
    "audit_standards_인증업무개념체계_2022": {
        "label": "FRMK-1",
        "expected_points": 121,
        "expected_summary_points": 1,
    },
}

N_STANDARDS = 3
K_PER_STANDARD = 5
MAX_CONTEXT_CHARS = 12000
CHUNK_CHAR_LIMIT = 1200

DEFAULT_ANTHROPIC_MODEL = "claude-sonnet-4-20250514"
ANTHROPIC_API_KEY = (os.environ.get("ANTHROPIC_API_KEY") or "").strip()
ANTHROPIC_MODEL = (os.environ.get("ANTHROPIC_MODEL") or DEFAULT_ANTHROPIC_MODEL).strip()
ANTHROPIC_BASE_URL = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com")

if ANTHROPIC_MODEL.startswith("sk-ant-"):
    raise ValueError(
        "ANTHROPIC_MODEL에 API key가 들어가 있습니다. "
        "키는 ANTHROPIC_API_KEY에 넣고, "
        f"ANTHROPIC_MODEL은 {DEFAULT_ANTHROPIC_MODEL!r}처럼 모델명으로 설정하세요."
    )
if ANTHROPIC_API_KEY and not ANTHROPIC_API_KEY.startswith("sk-ant-"):
    raise ValueError(
        "ANTHROPIC_API_KEY가 sk-ant-로 시작하지 않습니다. "
        "ANTHROPIC_API_KEY와 ANTHROPIC_MODEL 값이 뒤바뀌었는지 확인하세요."
    )

RUN_LLM = bool(ANTHROPIC_API_KEY)
LLM_MODEL = ANTHROPIC_MODEL

TEST_QUERIES = [
    "품질관리시스템의 궁극적인 책임은 누구에게 있는가?",
    "회계추정치 감사에서 경영진 추정치를 어떻게 평가해야 하는가?",
    "제한적 확신업무 결론은 어떻게 표현되는가?",
    "인증업무에서 세 당사자의 역할은 무엇인가?",
]

print(f"RUN_LLM={RUN_LLM}, provider=anthropic, model={LLM_MODEL}")


RUN_LLM=True, provider=anthropic, model=claude-sonnet-4-6


## 3. Qdrant Sanity Check

4개 collection의 point count, named vectors, payload index를 확인합니다.

In [3]:
summary_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="kind",
            match=models.MatchValue(value=KIND_STANDARD_SUMMARY),
        )
    ]
)

for collection, cfg in COLLECTIONS.items():
    info = client.get_collection(collection)
    vectors = info.config.params.vectors
    summary_count = client.count(
        collection_name=collection,
        count_filter=summary_filter,
        exact=True,
    ).count
    print(f"\n[{cfg['label']}] {collection}")
    print(f"  points:        {info.points_count} / {cfg['expected_points']}")
    print(f"  summaries:     {summary_count} / {cfg['expected_summary_points']}")
    print(f"  status:        {info.status} / {info.optimizer_status}")
    print(f"  vector names:  {sorted(vectors.keys())}")
    print(f"  payload index: {len(info.payload_schema)}")
    assert info.points_count == cfg["expected_points"]
    assert summary_count == cfg["expected_summary_points"]
    assert set(vectors) == {VECTOR_PASSAGE, VECTOR_SUMMARY}
    assert len(info.payload_schema) >= 12

print("\nOK — 4개 collection 기준선 확인 완료")



[ISA 36종] audit_standards_회계감사기준_2025
  points:        8626 / 8626
  summaries:     36 / 36
  status:        green / ok
  vector names:  ['passage', 'summary']
  payload index: 12

[ISQM-1] audit_standards_품질관리기준서_2018
  points:        400 / 400
  summaries:     1 / 1
  status:        green / ok
  vector names:  ['passage', 'summary']
  payload index: 12

[ASSR-3000] audit_standards_기타인증업무기준_2022
  points:        1240 / 1240
  summaries:     1 / 1
  status:        green / ok
  vector names:  ['passage', 'summary']
  payload index: 12

[FRMK-1] audit_standards_인증업무개념체계_2022
  points:        121 / 121
  summaries:     1 / 1
  status:        green / ok
  vector names:  ['passage', 'summary']
  payload index: 12

OK — 4개 collection 기준선 확인 완료


## 4. Anthropic Model Check

404가 발생하면 대부분 `ANTHROPIC_MODEL`이 실제 API model id와 맞지 않는 경우입니다. 먼저 `/v1/models/{model}`로 모델 접근성을 확인합니다.

In [4]:
def anthropic_headers() -> dict[str, str]:
    if not ANTHROPIC_API_KEY:
        return {}
    return {
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }


def check_anthropic_model(model: str = ANTHROPIC_MODEL) -> bool:
    if not ANTHROPIC_API_KEY:
        print("ANTHROPIC_API_KEY 미설정 — model check 생략")
        return False
    url = f"{ANTHROPIC_BASE_URL.rstrip('/')}/v1/models/{model}"
    response = httpx.get(url, headers=anthropic_headers(), timeout=30)
    if response.status_code == 200:
        data = response.json()
        print(f"Anthropic model OK: {data.get('id')} ({data.get('display_name')})")
        return True
    print(f"Anthropic model check failed: HTTP {response.status_code}")
    print(response.text[:1000])
    return False


MODEL_OK = check_anthropic_model()


Anthropic model OK: claude-sonnet-4-6 (Claude Sonnet 4.6)


## 5. 검색 Helper

Qdrant `query_points()` 호출 결과를 notebook에서 보기 쉬운 dict로 정규화합니다.

In [5]:
def embed_query(query_text: str) -> tuple[float, ...]:
    return embedder.embed_query(query_text).vector


def hit_to_dict(point: Any, *, collection: str, rank: int) -> dict[str, Any]:
    payload = point.payload or {}
    text = str(payload.get("content_text") or "")
    return {
        "rank": rank,
        "score": round(float(point.score), 6),
        "collection": collection,
        "standard_id": payload.get("standard_id"),
        "chunk_id": payload.get("chunk_id"),
        "kind": payload.get("kind"),
        "section": payload.get("section"),
        "paragraph_id": payload.get("paragraph_id"),
        "heading_trail": payload.get("heading_trail") or [],
        "special_appendix_name": payload.get("special_appendix_name"),
        "content_text": text,
        "content_preview": text[:220].replace("\n", " "),
    }


def qdrant_search(
    *,
    collection: str,
    query_vector: tuple[float, ...],
    using: str,
    limit: int,
    query_filter: models.Filter | None = None,
) -> list[dict[str, Any]]:
    response = client.query_points(
        collection_name=collection,
        query=list(query_vector),
        using=using,
        query_filter=query_filter,
        limit=limit,
        with_payload=True,
        with_vectors=False,
    )
    return [
        hit_to_dict(point, collection=collection, rank=i)
        for i, point in enumerate(response.points, start=1)
    ]


## 6. Step 1 — 기준서 식별

각 collection의 `standard_summary` point를 `summary` vector로 검색하고 전체 후보를 score 순으로 정렬합니다.

In [6]:
def identify_standards(query_text: str, top_n: int = N_STANDARDS) -> list[dict[str, Any]]:
    qvec = embed_query(query_text)
    candidates: list[dict[str, Any]] = []
    for collection in COLLECTIONS:
        hits = qdrant_search(
            collection=collection,
            query_vector=qvec,
            using=VECTOR_SUMMARY,
            limit=top_n,
            query_filter=summary_filter,
        )
        candidates.extend(hits)
    candidates.sort(key=lambda item: item["score"], reverse=True)
    selected = candidates[:top_n]

    print(f"Query: {query_text}\n")
    print(f"{'rank':<5} {'score':<9} {'standard':<12} collection")
    print("-" * 80)
    for i, row in enumerate(selected, start=1):
        row["rank"] = i
        print(f"{i:<5} {row['score']:<9.4f} {row['standard_id']:<12} {row['collection']}")
    return selected


standards = identify_standards(TEST_QUERIES[0])


Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

rank  score     standard     collection
--------------------------------------------------------------------------------
1     0.2993    ISA-220      audit_standards_회계감사기준_2025
2     0.2546    ISA-200      audit_standards_회계감사기준_2025
3     0.2533    ISA-1200     audit_standards_회계감사기준_2025


## 7. Step 2 — Passage 검색

Step 1의 후보 `standard_id`별로 `passage` vector를 검색합니다.

In [7]:
def search_passages(
    query_text: str,
    standards: list[dict[str, Any]],
    k_per_standard: int = K_PER_STANDARD,
) -> list[dict[str, Any]]:
    qvec = embed_query(query_text)
    all_hits: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for candidate in standards:
        collection = str(candidate["collection"])
        standard_id = str(candidate["standard_id"])
        key = (collection, standard_id)
        if key in seen:
            continue
        seen.add(key)
        filter_ = models.Filter(
            must=[
                models.FieldCondition(
                    key="standard_id",
                    match=models.MatchValue(value=standard_id),
                )
            ]
        )
        hits = qdrant_search(
            collection=collection,
            query_vector=qvec,
            using=VECTOR_PASSAGE,
            limit=k_per_standard,
            query_filter=filter_,
        )
        all_hits.extend(hits)

    all_hits.sort(key=lambda item: item["score"], reverse=True)
    for i, row in enumerate(all_hits, start=1):
        row["global_rank"] = i

    print(f"Query: {query_text}\n")
    for row in all_hits[: k_per_standard * max(1, len(seen))]:
        print(
            f"#{row['global_rank']:<2} score={row['score']:.4f} "
            f"{row['standard_id']} {row['chunk_id']}"
        )
        print(f"    {row['content_preview']}...")
    return all_hits


passage_hits = search_passages(TEST_QUERIES[0], standards)


Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

#1  score=0.3712 ISA-220 ISA-220:intro:eab83888:2.
    품질관리시스템, 정책과 절차는 회계법인의 책임이다. 회계법인은 품질관리기준서1에 따라 다음에 대하여 합리적 확신을 제공할 수 있도록 품질관리시스템을 수립하고 유지할 의무가 있다....
#2  score=0.3419 ISA-220 ISA-220:application:dc8737e7:bullet#702
    회계법인의 내 품질에 대한 리더십 책임...
#3  score=0.3388 ISA-220 ISA-220:intro:eab83888:3.
    업무팀은 회계법인의 품질관리시스템 관점에서 감사업무에 적용가능한 품질관리절차를 실행하고 품질관리시스템 중 독립성과 관련된 부분이 기능할 수 있도록 관련 정보를 회계법인에게 제공할 책임이 있다....
#4  score=0.3383 ISA-220 ISA-220:application:2acb3ede:A32.
    공공부문에서는 법률이 정하는 바에 따라 선임된 감사인 (예를 들어, 정부의 감사부서 책임자 또는 이를 대신하여 선임된 적격한 자격을 갖춘 사람)이 공공부문감사에 대한 전반적인 책임에 있어 업무수행이사와 동등한 역할을 수행할 수 있다. 그러한 상황에서, 해당되는 경우 업무품질관리검토자를 선정할 때는 감사대상으로부터 독립적이어야 할 필요성 그리고 객관적인 평가를 제공할 수 있는 업무품질관리검토자...
#5  score=0.3369 ISA-1200 ISA-1200:general_principles:5c7ad26d:18.
    업무수행이사는 개별 감사업무의 전반적인 품질에 책임을 진다....
#6  score=0.3334 ISA-220 ISA-220:intro:2adefd7b:1.
    이 감사기준서는 재무제표감사의 품질관리 절차에 관한 감사인의 구체적인 책임을 다룬다. 또한 해당되는 경우, 업무 품질관리검토자의 책임도 다룬다. 이 감사기준서는 관련 윤리적

## 8. Step 3 — LLM Context Builder

LLM에는 `chunk_id` citation을 포함한 제한된 컨텍스트만 전달합니다.

In [8]:
def build_llm_context(
    query_text: str,
    standards: list[dict[str, Any]],
    passage_hits: list[dict[str, Any]],
    *,
    max_chars: int = MAX_CONTEXT_CHARS,
    chunk_char_limit: int = CHUNK_CHAR_LIMIT,
) -> str:
    lines: list[str] = []
    lines.append("# Audit Standards RAG Context")
    lines.append(f"사용자 질문: {query_text}")
    lines.append("")

    lines.append("## 1. 기준서 후보")
    for row in standards:
        lines.append(
            f"- score={row['score']:.4f} standard_id={row['standard_id']} "
            f"collection={row['collection']}"
        )
        preview = row["content_text"][:500].replace("\n", " ")
        if preview:
            lines.append(f"  summary: {preview}")
    lines.append("")

    lines.append("## 2. 검색된 근거 문단")
    for row in passage_hits:
        heading = " > ".join(str(v) for v in row.get("heading_trail") or [] if v)
        special = row.get("special_appendix_name")
        meta = [
            f"score={row['score']:.4f}",
            f"standard_id={row['standard_id']}",
            f"kind={row['kind']}",
        ]
        if row.get("section"):
            meta.append(f"section={row['section']}")
        if row.get("paragraph_id"):
            meta.append(f"paragraph_id={row['paragraph_id']}")
        if special:
            meta.append(f"special_appendix_name={special}")
        lines.append(f"\n### [{row['chunk_id']}]")
        lines.append("; ".join(meta))
        if heading:
            lines.append(f"heading: {heading}")
        lines.append(row["content_text"][:chunk_char_limit])

    context = "\n".join(lines)
    if len(context) > max_chars:
        context = context[:max_chars] + "\n\n[TRUNCATED]"
    print(f"context chars: {len(context):,}")
    return context


context = build_llm_context(TEST_QUERIES[0], standards, passage_hits)
print(context[:2500])


context chars: 6,269
# Audit Standards RAG Context
사용자 질문: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

## 1. 기준서 후보
- score=0.2993 standard_id=ISA-220 collection=audit_standards_회계감사기준_2025
  summary: 1. 이 감사기준서는 재무제표감사의 품질관리 절차에 관한 감사인의 구체적인 책임을 다룬다. 또한 해당되는 경우, 업무 품질관리검토자의 책임도 다룬다. 이 감사기준서는 관련 윤리적 요구사항과 함께 이해하여야 한다.  7. 감사기준에서 사용하는 용어의 정의는 다음과 같다.   (a) 업무수행이사 - 감사업무와 감사업무 수행 및 발행된 감사보고서에 대하여 회계법인을 대신하여 책임을 지며, 관련 요구사항이 있을 경우에는 전문직단체, 정부 또는 규제기관으로부터 적합한 권한을 받은 파트너 또는 회계법인 내 기타의 사람   (b) 업무품질관리검토 – 감사보고서를 작성할 때 업무팀이 내린 유의적 판단 및 그 도달한 결론에 대하여 감사보고서일 또는 그 전에 객관적인 평가를 제공하기 위하여 설계된 절차. 업무품질관리검토 절차는 상장기업의 재무제표감사 그리고 해당 회계법인이 품질관리검토가 필요하다고 결정한 감사업무를 대상으로 하는 것이다.   (c) 업무품질관리검토자 – 업무팀에 소속되어 있지 않으며,
- score=0.2546 standard_id=ISA-200 collection=audit_standards_회계감사기준_2025
  summary: 1. 이 감사기준서는 독립된 감사인이 감사기준 (이 감사기준서를 포함한 전체 감사기준서를 총칭하여 “감사기준”이라 한다)에 따라 재무제표감사를 수행할 때 전반적인 책임을 다룬다. 이 감사기준서는 특히 독립된 감사인의 전반적인 목적을 제시하며, 독립된 감사인이 해당 목적을 충족할 수 있도록 설계된 감사의 성격과 범위를 설명한다. 또한, 이 감사기준서는 감사기준의 범위, 권위 및 구조를 설명하고, 감사기준의 

## 9. Step 4 — Anthropic 답변

`ANTHROPIC_API_KEY`가 있으면 Anthropic Messages API로 답변을 생성합니다. 키가 없으면 LLM 호출을 건너뛰고 검색/컨텍스트만 검증합니다.

In [9]:
SYSTEM_PROMPT = """
너는 회계감사기준 검색 결과를 바탕으로 답변하는 보조자다.
반드시 제공된 컨텍스트 안의 정보만 사용한다.
답변의 핵심 문장마다 근거 chunk_id를 대괄호로 표시한다.
컨텍스트에서 답을 찾을 수 없으면 추정하지 말고 '제공된 근거만으로는 확인할 수 없습니다'라고 답한다.
기준서 원문과 해석을 구분하고, 법률/감사 자문처럼 단정하지 않는다.
""".strip()


def answer_with_anthropic(
    query_text: str,
    context: str,
    *,
    model: str = ANTHROPIC_MODEL,
    max_tokens: int = 1200,
) -> str | None:
    if not ANTHROPIC_API_KEY:
        print("ANTHROPIC_API_KEY 미설정 — LLM 호출을 건너뜁니다.")
        return None
    if model.startswith("sk-ant-"):
        raise ValueError("model 인자에 API key가 들어갔습니다. ANTHROPIC_MODEL 환경변수를 확인하세요.")
    url = f"{ANTHROPIC_BASE_URL.rstrip('/')}/v1/messages"
    response = httpx.post(
        url,
        headers=anthropic_headers(),
        json={
            "model": model,
            "max_tokens": max_tokens,
            "temperature": 0,
            "system": SYSTEM_PROMPT,
            "messages": [
                {
                    "role": "user",
                    "content": f"질문:\n{query_text}\n\n컨텍스트:\n{context}",
                }
            ],
        },
        timeout=90,
    )
    if response.status_code >= 400:
        raise RuntimeError(
            f"Anthropic API HTTP {response.status_code}; "
            f"model={model}; url={url}; body={response.text[:1000]}"
        )
    data = response.json()
    text_blocks = [
        block.get("text", "")
        for block in data.get("content", [])
        if block.get("type") == "text"
    ]
    return "\n".join(text_blocks).strip()


def answer_with_llm(query_text: str, context: str) -> str | None:
    return answer_with_anthropic(query_text, context, model=ANTHROPIC_MODEL)


answer = answer_with_llm(TEST_QUERIES[0], context) if RUN_LLM else None
print(answer or "LLM 호출 생략 — RUN_LLM=False 또는 ANTHROPIC_API_KEY 미설정")


## 품질관리시스템의 궁극적인 책임 주체

### 1. 회계법인의 책임 (시스템 수준)

품질관리시스템 전반에 대한 궁극적인 책임은 **회계법인**에 있습니다.

> "품질관리시스템, 정책과 절차는 **회계법인의 책임**이다. 회계법인은 품질관리기준서1에 따라 합리적 확신을 제공할 수 있도록 품질관리시스템을 수립하고 유지할 의무가 있다." [ISA-220:intro:eab83888:2.]

이는 ISA-200의 적용지침에서도 확인됩니다.

> "품질관리기준서1은 해당 회계법인 및 소속 구성원이 독립성 등 관련 윤리적 요구사항을 준수하고 있다는 합리적 확신을 제공할 수 있도록 정책과 절차를 수립해야 하는 **회계법인의 책임**을 정한다." [ISA-200:application:79491974:A20.]

---

### 2. 업무수행이사의 책임 (개별 감사업무 수준)

개별 감사업무 차원에서는 **업무수행이사**가 전반적인 품질에 책임을 집니다.

> "업무수행이사는 **개별 감사업무의 전반적인 품질**에 책임을 진다." [ISA-1200:general_principles:5c7ad26d:18.]

---

### 3. 업무팀의 책임 (실행 수준)

업무팀은 시스템을 직접 운영·실행하는 역할을 담당합니다.

> "업무팀은 회계법인의 품질관리시스템 관점에서 감사업무에 적용가능한 품질관리절차를 **실행**하고, 독립성과 관련된 부분이 기능할 수 있도록 관련 정보를 회계법인에게 **제공**할 책임이 있다." [ISA-220:intro:eab83888:3.]

---

### 요약 정리

| 주체 | 책임 범위 |
|------|-----------|
| **회계법인** | 품질관리시스템 전반의 수립·유지 (궁극적 책임) |
| **업무수행이사** | 개별 감사업무의 전반적 품질 |
| **업무팀** | 품질관리절차의 실행 및 관련 정보 제공 |

> ※ 위 내용은 감사기준서 220 및 1200의 원문에 근거한 것이며, 구체적인 법률·감사 자문으로 해석되어서는 안 됩니

## 10. End-to-End Pipeline

`rag_pipeline(query)` 하나로 기준서 식별, passage 검색, context 구성, Anthropic 답변을 실행합니다.

In [10]:
def rag_pipeline(
    query_text: str,
    *,
    n_standards: int = N_STANDARDS,
    k_per_standard: int = K_PER_STANDARD,
    run_llm: bool = RUN_LLM,
) -> dict[str, Any]:
    started = time.perf_counter()
    print("=" * 90)
    print(f"QUERY: {query_text}")
    print("=" * 90)
    standards = identify_standards(query_text, top_n=n_standards)
    passages = search_passages(query_text, standards, k_per_standard=k_per_standard)
    context = build_llm_context(query_text, standards, passages)
    answer = answer_with_llm(query_text, context) if run_llm else None
    elapsed = time.perf_counter() - started

    if answer:
        print("\nANTHROPIC ANSWER")
        print("-" * 90)
        print(answer)
    else:
        print("\nANTHROPIC ANSWER: skipped")

    return {
        "query": query_text,
        "elapsed_seconds": round(elapsed, 3),
        "standards": standards,
        "passages": passages,
        "context": context,
        "answer": answer,
        "llm_provider": "anthropic" if answer else None,
        "llm_model": LLM_MODEL if answer else None,
    }


result = rag_pipeline(TEST_QUERIES[0], run_llm=RUN_LLM)


QUERY: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?
Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

rank  score     standard     collection
--------------------------------------------------------------------------------
1     0.2993    ISA-220      audit_standards_회계감사기준_2025
2     0.2546    ISA-200      audit_standards_회계감사기준_2025
3     0.2533    ISA-1200     audit_standards_회계감사기준_2025
Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

#1  score=0.3712 ISA-220 ISA-220:intro:eab83888:2.
    품질관리시스템, 정책과 절차는 회계법인의 책임이다. 회계법인은 품질관리기준서1에 따라 다음에 대하여 합리적 확신을 제공할 수 있도록 품질관리시스템을 수립하고 유지할 의무가 있다....
#2  score=0.3419 ISA-220 ISA-220:application:dc8737e7:bullet#702
    회계법인의 내 품질에 대한 리더십 책임...
#3  score=0.3388 ISA-220 ISA-220:intro:eab83888:3.
    업무팀은 회계법인의 품질관리시스템 관점에서 감사업무에 적용가능한 품질관리절차를 실행하고 품질관리시스템 중 독립성과 관련된 부분이 기능할 수 있도록 관련 정보를 회계법인에게 제공할 책임이 있다....
#4  score=0.3383 ISA-220 ISA-220:application:2acb3ede:A32.
    공공부문에서는 법률이 정하는 바에 따라 선임된 감사인 (예를 들어, 정부의 감사부서 책임자 또는 이를 대신하여 선임된 적격한 자격을 갖춘 사람)이 공공부문감사에 대한 전반적인 책임에 있어 업무수행이사와 동등한 역할을

## 11. Batch Smoke Test

기본은 검색/컨텍스트만 검증합니다. Anthropic 비용을 의도적으로 쓰려면 `RUN_LLM_BATCH=True`로 바꾸세요.

In [11]:
RUN_LLM_BATCH = False

batch_results: list[dict[str, Any]] = []
for query in TEST_QUERIES:
    batch_results.append(rag_pipeline(query, run_llm=RUN_LLM_BATCH))

summary_rows = []
for item in batch_results:
    top_standard = item["standards"][0]["standard_id"] if item["standards"] else None
    top_chunk = item["passages"][0]["chunk_id"] if item["passages"] else None
    summary_rows.append(
        {
            "query": item["query"],
            "top_standard": top_standard,
            "top_chunk": top_chunk,
            "answer_generated": item["answer"] is not None,
        }
    )

summary_rows


QUERY: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?
Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

rank  score     standard     collection
--------------------------------------------------------------------------------
1     0.2993    ISA-220      audit_standards_회계감사기준_2025
2     0.2546    ISA-200      audit_standards_회계감사기준_2025
3     0.2533    ISA-1200     audit_standards_회계감사기준_2025
Query: 품질관리시스템의 궁극적인 책임은 누구에게 있는가?

#1  score=0.3712 ISA-220 ISA-220:intro:eab83888:2.
    품질관리시스템, 정책과 절차는 회계법인의 책임이다. 회계법인은 품질관리기준서1에 따라 다음에 대하여 합리적 확신을 제공할 수 있도록 품질관리시스템을 수립하고 유지할 의무가 있다....
#2  score=0.3419 ISA-220 ISA-220:application:dc8737e7:bullet#702
    회계법인의 내 품질에 대한 리더십 책임...
#3  score=0.3388 ISA-220 ISA-220:intro:eab83888:3.
    업무팀은 회계법인의 품질관리시스템 관점에서 감사업무에 적용가능한 품질관리절차를 실행하고 품질관리시스템 중 독립성과 관련된 부분이 기능할 수 있도록 관련 정보를 회계법인에게 제공할 책임이 있다....
#4  score=0.3383 ISA-220 ISA-220:application:2acb3ede:A32.
    공공부문에서는 법률이 정하는 바에 따라 선임된 감사인 (예를 들어, 정부의 감사부서 책임자 또는 이를 대신하여 선임된 적격한 자격을 갖춘 사람)이 공공부문감사에 대한 전반적인 책임에 있어 업무수행이사와 동등한 역할을

[{'query': '품질관리시스템의 궁극적인 책임은 누구에게 있는가?',
  'top_standard': 'ISA-220',
  'top_chunk': 'ISA-220:intro:eab83888:2.',
  'answer_generated': False},
 {'query': '회계추정치 감사에서 경영진 추정치를 어떻게 평가해야 하는가?',
  'top_standard': 'ISA-540',
  'top_chunk': 'ISA-540:requirements:932701fe:36.',
  'answer_generated': False},
 {'query': '제한적 확신업무 결론은 어떻게 표현되는가?',
  'top_standard': 'ISA-1200',
  'top_chunk': 'ISA-200:application:00425dd0:A50.',
  'answer_generated': False},
 {'query': '인증업무에서 세 당사자의 역할은 무엇인가?',
  'top_standard': 'ISA-402',
  'top_chunk': 'ISA-402:application:ef466fa3:A27.',
  'answer_generated': False}]

## 12. 결과 저장

`output/`은 gitignored입니다. notebook 실행 결과를 로컬에 남겨 비교할 때 사용하세요.

In [12]:
out_path = PROJECT_ROOT / "output" / "phase4_llm_rag_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

serializable = []
for item in batch_results:
    slim = dict(item)
    slim["context"] = item["context"][:2000]
    serializable.append(slim)

out_path.write_text(
    json.dumps(serializable, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"saved: {out_path}")


saved: /home/shin/Project/_AuditStandard_parsing/output/phase4_llm_rag_results.json


## 13. Cleanup

Notebook 세션을 오래 유지할 경우 SQLite cache 연결을 명시적으로 닫습니다.

In [13]:
embedder.close()
print("Embedder cache connection closed")


Embedder cache connection closed
